# Quantize Ministral 3 for MLX

This notebook converts Mistral AI's multimodal **Ministral 3 3B Instruct** checkpoint from Hugging Face BF16 Safetensors into a 4-bit MLX checkpoint. It then runs text generation and, when a local image is supplied, a vision smoke test.

The source is deliberately the Hugging Face checkpoint rather than an Ollama/GGUF file. MLX conversion preserves the model configuration, tokenizer, processor, and vision components while quantizing the language-model weights.

## Learning objectives

By the end of this lesson you will be able to:

1. distinguish an Ollama/GGUF distribution from an MLX checkpoint;
2. convert a Hugging Face multimodal checkpoint with `mlx_vlm.convert`;
3. choose a 4-bit quantization configuration and inspect the resulting artifact;
4. validate the converted model with text and optional image inference.

In [1]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

if platform.system() != 'Darwin' or platform.machine() != 'arm64':
    raise RuntimeError('This lesson requires an Apple Silicon Mac running macOS.')

MODULE_DIR = Path.cwd()
if MODULE_DIR.name != '20-mlx-quantization':
    MODULE_DIR = Path('20-mlx-quantization').resolve()
ARTIFACT_DIR = MODULE_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'mistralai/Ministral-3-3B-Instruct-2512-BF16'
OUTPUT_DIR = ARTIFACT_DIR / 'Ministral-3-3B-Instruct-2512-4bit'
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.machine()} {platform.mac_ver()[0]}')
print(f'Source: {MODEL_ID}')
print(f'Output: {OUTPUT_DIR}')
print(f'Existing output: {OUTPUT_DIR.exists()}')

Python: 3.13.15
Platform: arm64 26.4
Source: mistralai/Ministral-3-3B-Instruct-2512-BF16
Output: /Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/artifacts/Ministral-3-3B-Instruct-2512-4bit
Existing output: True


## 1. Convert and quantize

A 4-bit affine quantization with group size 64 is a practical starting point on a 16 GB Apple Silicon machine. The conversion is skipped when the output already contains a configuration file, so re-running the notebook is safe.

In [2]:
if not (OUTPUT_DIR / 'config.json').exists():
    command = [
        'mlx_vlm.convert',
        '--hf-path', MODEL_ID,
        '--mlx-path', str(OUTPUT_DIR),
        '--quantize',
        '--q-bits', '4',
        '--q-group-size', '64',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True, cwd=MODULE_DIR)
else:
    print('A converted checkpoint already exists; skipping conversion.')

# Current Transformers releases can misread this generated file for Mistral tokenizers.
# The tokenizer configuration itself remains intact; removing this optional map keeps
# both mlx-lm and mlx-vlm compatible with the converted checkpoint.
special_tokens_map = OUTPUT_DIR / 'special_tokens_map.json'
if special_tokens_map.exists():
    special_tokens_map.unlink()
    print('Removed optional special_tokens_map.json for tokenizer compatibility.')

weights = sorted(OUTPUT_DIR.glob('*.safetensors'))
size_gib = sum(path.stat().st_size for path in weights) / 2**30
print(f'Converted weight files: {len(weights)}')
print(f'Converted weight size: {size_gib:.2f} GiB')

A converted checkpoint already exists; skipping conversion.
Converted weight files: 1
Converted weight size: 2.58 GiB


## 2. Text inference smoke test

The command below loads the local MLX checkpoint and verifies that its tokenizer, chat template, quantized weights, and generation path work together.

In [3]:
text_command = [
    sys.executable, '-m', 'mlx_lm', 'generate',
    '--model', str(OUTPUT_DIR),
    '--prompt', 'Explain in three sentences why quantization makes local inference cheaper.',
    '--max-tokens', '120',
    '--temp', '0',
]
print('Running:', ' '.join(text_command))
subprocess.run(text_command, check=True, cwd=MODULE_DIR)

Running: /Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/.venv/bin/python -m mlx_lm generate --model /Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/artifacts/Ministral-3-3B-Instruct-2512-4bit --prompt Explain in three sentences why quantization makes local inference cheaper. --max-tokens 120 --temp 0


[transformers] The tokenizer you are loading from '/Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/artifacts/Ministral-3-3B-Instruct-2512-4bit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Quantization reduces the computational cost of local inference by

 replacing high-precision floating-point operations (e.g

., 32-bit or 64-bit

...

 consumption. Additionally, quantized models often fit more

 compactly in hardware accelerators (like GPUs

 or TPUs), enabling faster execution while maintaining reasonable

 accuracy for many applications.
Prompt: 538 tokens, 522.074 tokens-per-sec
Generation: 116 tokens, 46.922 tokens-per-sec
Peak memory: 2.458 GB


CompletedProcess(args=['/Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/.venv/bin/python', '-m', 'mlx_lm', 'generate', '--model', '/Users/marco/GitHub/LLM-crash-course/20-mlx-quantization/artifacts/Ministral-3-3B-Instruct-2512-4bit', '--prompt', 'Explain in three sentences why quantization makes local inference cheaper.', '--max-tokens', '120', '--temp', '0'], returncode=0)

## 3. Optional vision inference

Set `IMAGE_PATH` to an existing local image to exercise the multimodal processor. Leaving it as `None` keeps this cell runnable on a clean checkout.

In [4]:
IMAGE_PATH = None  # Example: MODULE_DIR / 'assets' / 'example.jpg'
if IMAGE_PATH is None:
    print('Vision test skipped: set IMAGE_PATH to a local image to run it.')
else:
    image_path = Path(IMAGE_PATH)
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    vision_command = [
        sys.executable, '-m', 'mlx_vlm.generate',
        '--model', str(OUTPUT_DIR),
        '--image', str(image_path),
        '--prompt', 'Describe this image briefly.',
        '--max-tokens', '120',
        '--temperature', '0',
    ]
    subprocess.run(vision_command, check=True, cwd=MODULE_DIR)

Vision test skipped: set IMAGE_PATH to a local image to run it.


## Takeaways

The MLX artifact is independent of Ollama: it contains MLX Safetensors plus the configuration and processor files needed by `mlx-vlm`. Lowering `--q-bits` reduces memory further but can reduce quality; test the intended workload before publishing a quantized checkpoint.